In [1]:
from collections import Counter
import xml.etree.ElementTree as ET
tree = ET.parse('Block_1.xml')
root = tree.getroot()

In [2]:
''' O usuario fornecera os dados da seguinte forma[
[(Nome_elemento,tipo_elemento,subtipo_elemento,0),aresta_anterior, aresta_posterior]
[(Contato1,contato,1,0),0,1]
A aresta anterior deverá ser zero SEMPRE que o elemento estiver diretamente ligado ao barramento energizado
]'''
def diferenciar_nomes(dados_ladder):
    '''Funçao de pre processamento para retirar nomes repetidos e ordenar o problema'''
    contador = {}
    for x in dados_ladder:
        elemento = x[0]
        if elemento in contador:
            contador[elemento] +=1
        else:
            contador[elemento] = 1
        repeticoes = contador[elemento] -1
        x[0] = x[0][:-1] + (1*repeticoes,)
    return(dados_ladder)

In [3]:
def gerar_chaves_ladder(lista_elementos):
    elementos_dict = {}
    
    # ==========================================
    # 1. REGISTRO (Agora com apenas 3 informações por lista)
    # Formato esperado: [Nome_do_Elemento, No_Anterior, No_Posterior]
    # ==========================================
    for i, el in enumerate(lista_elementos):
        elementos_dict[i] = {
            "nome": el[0], 
            "no_in": el[1], 
            "no_out": el[2],
            "predecessores": set(), 
            "sucessores": set(),
            "indice_original": i,
            "y_score": None # A Linha Virtual
        }

    # ==========================================
    # 2. MAPEAMENTO DOS FIOS (O Grafo)
    # ==========================================
    saidas_por_no = {}
    for i, el in elementos_dict.items():
        if el["no_out"] not in saidas_por_no: 
            saidas_por_no[el["no_out"]] = []
        saidas_por_no[el["no_out"]].append(i)

    for i, el in elementos_dict.items():
        if el["no_in"] in saidas_por_no:
            for pred_id in saidas_por_no[el["no_in"]]:
                el["predecessores"].add(pred_id)
                elementos_dict[pred_id]["sucessores"].add(i)

    # ==========================================
    # 3. CÁLCULO DA LINHA VIRTUAL (De cima para baixo)
    # ==========================================
    # Pega quem está ligado direto na barra (Nó 0) e ordena pela digitação
    raizes = [i for i, el in elementos_dict.items() if len(el["predecessores"]) == 0]
    raizes.sort(key=lambda x: elementos_dict[x]["indice_original"])
    
    # Dá a eles uma "linha" base (0, 1, 2, 3...)
    for linha, id_raiz in enumerate(raizes):
        elementos_dict[id_raiz]["y_score"] = linha

    # Propaga essa "linha" para frente no circuito
    grau_entrada_y = {i: len(el["predecessores"]) for i, el in elementos_dict.items()}
    fila_y = raizes[:]
    
    while fila_y:
        atual_id = fila_y.pop(0)
        atual = elementos_dict[atual_id]
        
        for suc_id in atual["sucessores"]:
            suc = elementos_dict[suc_id]
            
            # O sucessor herda a linha da raiz mais alta
            if suc["y_score"] is None:
                suc["y_score"] = atual["y_score"]
            else:
                suc["y_score"] = min(suc["y_score"], atual["y_score"])
            
            grau_entrada_y[suc_id] -= 1
            if grau_entrada_y[suc_id] == 0:
                fila_y.append(suc_id)

    # ==========================================
    # 4. ORDENAÇÃO TOPOLÓGICA (A Esteira de Numeração)
    # ==========================================
    grau_entrada = {i: len(el["predecessores"]) for i, el in elementos_dict.items()}
    prontos = [i for i, el in elementos_dict.items() if grau_entrada[i] == 0]
    
    resultado = []
    contador = 1
    
    while prontos:
        # A Regra de Ouro: Ordena pela Linha Virtual (y_score) e desempata pelo Índice Original
        prontos.sort(key=lambda x: (elementos_dict[x]["y_score"], elementos_dict[x]["indice_original"]))
        
        atual_id = prontos.pop(0)
        atual = elementos_dict[atual_id]
        
        resultado.append({
            "Chave": contador, 
            "Elemento": atual["nome"], 
            "Linha_Virtual": atual["y_score"],
            "No_in": atual["no_in"],
            "No_out": atual["no_out"]
        })
        contador += 1
        
        for suc_id in atual["sucessores"]:
            grau_entrada[suc_id] -= 1
            if grau_entrada[suc_id] == 0:
                prontos.append(suc_id)
                
    return resultado


In [5]:
# ==========================================
# ÁREA DE TESTE
# ==========================================
# Formato Otimizado: [Elemento, Nó_Anterior, Nó_Posterior]
# Vou manter a lista bagunçada para provar que a matemática organiza sozinha!
dados_ladder_otimizados = [
    ["cont1", 0, 1],
    ["cont3", 0, 2],
    ["cont4", 0, 2],
    ["cont9_inf", 0, 4],
    ["cont9_sup", 1, 2],
    ["cont10", 4, 3],
    ["cont6", 3, 5],
    ["bobina1", 5, 6],
    ["cont7", 3, 7],
    ["cont8", 7, 8],
    ["bobina2", 8, 9],
    ["cont5", 2, 3]  # <--- O elemento central jogado no final da lista
]

# Executando a função
elementos_nomeados = gerar_chaves_ladder(dados_ladder_otimizados)

'''# Imprimindo o resultado formatado
print(f"{'CHAVE':<7} | {'ELEMENTO':<12} | {'LINHA VIRTUAL':<15} | {'CONEXÃO (In -> Out)'}")
print("-" * 65)
for item in elementos_nomeados:
    print(f" {item['Chave']:02d}     | {item['Elemento']:<12} | Linha {item['Linha_Virtual']:<10} | Nó {item['No_in']} -> Nó {item['No_out']}")'''

'# Imprimindo o resultado formatado\nprint(f"{\'CHAVE\':<7} | {\'ELEMENTO\':<12} | {\'LINHA VIRTUAL\':<15} | {\'CONEXÃO (In -> Out)\'}")\nprint("-" * 65)\nfor item in elementos_nomeados:\n    print(f" {item[\'Chave\']:02d}     | {item[\'Elemento\']:<12} | Linha {item[\'Linha_Virtual\']:<10} | Nó {item[\'No_in\']} -> Nó {item[\'No_out\']}")'

In [7]:
def gerar_sequencia_ladder_com_nos(lista_elementos):
    elementos_dict = {}
    
    # 1. REGISTRO [Nome, No_in, No_out]
    for i, el in enumerate(lista_elementos):
        elementos_dict[i] = {
            "nome": el[0], 
            "no_in": el[1], 
            "no_out": el[2],
            "predecessores": set(), 
            "sucessores": set(),
            "indice_original": i,
            "y_score": None 
        }

    # 2. MAPEAMENTO DOS FIOS
    saidas_por_no = {}
    for i, el in elementos_dict.items():
        if el["no_out"] not in saidas_por_no: 
            saidas_por_no[el["no_out"]] = []
        saidas_por_no[el["no_out"]].append(i)

    for i, el in elementos_dict.items():
        if el["no_in"] in saidas_por_no:
            for pred_id in saidas_por_no[el["no_in"]]:
                el["predecessores"].add(pred_id)
                elementos_dict[pred_id]["sucessores"].add(i)

    # 3. CÁLCULO DA LINHA VIRTUAL
    raizes = [i for i, el in elementos_dict.items() if len(el["predecessores"]) == 0]
    raizes.sort(key=lambda x: elementos_dict[x]["indice_original"])
    
    for linha, id_raiz in enumerate(raizes):
        elementos_dict[id_raiz]["y_score"] = linha

    grau_entrada_y = {i: len(el["predecessores"]) for i, el in elementos_dict.items()}
    fila_y = raizes[:]
    
    while fila_y:
        atual_id = fila_y.pop(0)
        atual = elementos_dict[atual_id]
        
        for suc_id in atual["sucessores"]:
            suc = elementos_dict[suc_id]
            if suc["y_score"] is None:
                suc["y_score"] = atual["y_score"]
            else:
                suc["y_score"] = min(suc["y_score"], atual["y_score"])
            
            grau_entrada_y[suc_id] -= 1
            if grau_entrada_y[suc_id] == 0:
                fila_y.append(suc_id)

    # 4. ORDENAÇÃO TOPOLÓGICA
    grau_entrada = {i: len(el["predecessores"]) for i, el in elementos_dict.items()}
    prontos = [i for i, el in elementos_dict.items() if grau_entrada[i] == 0]
    
    resultado = [] 
    
    while prontos:
        prontos.sort(key=lambda x: (elementos_dict[x]["y_score"], elementos_dict[x]["indice_original"]))
        
        atual_id = prontos.pop(0)
        atual = elementos_dict[atual_id]
        
        # AQUI ESTÁ A MUDANÇA: Guarda a sub-lista completa com nome e nós
        resultado.append([atual["nome"], atual["no_in"], atual["no_out"]])
        
        for suc_id in atual["sucessores"]:
            grau_entrada[suc_id] -= 1
            if grau_entrada[suc_id] == 0:
                prontos.append(suc_id)
                
    return resultado

# ==========================================
# TESTE DA NOVA SAÍDA
# ==========================================
dados_ladder_otimizados = [
    ["cont1", 0, 1],
    ["cont3", 0, 2],
    ["cont4", 0, 2],
    ["cont9_inf", 0, 4],
    ["cont9_sup", 1, 2],
    ["cont10", 4, 3],
    ["cont6", 3, 5],
    ["bobina1", 5, 6],
    ["cont7", 3, 7],
    ["cont8", 7, 8],
    ["bobina2", 8, 9],
    ["cont5", 2, 3]  
]

lista_ordenada = gerar_sequencia_ladder_com_nos(dados_ladder_otimizados)

# Imprimindo item por item para visualizar melhor a estrutura
print("Sua lista final para o gerador de XML:")
print("[\n  " + ",\n  ".join(str(item) for item in lista_ordenada) + "\n]")

Sua lista final para o gerador de XML:
[
  ['cont1', 0, 1],
  ['cont9_sup', 1, 2],
  ['cont3', 0, 2],
  ['cont4', 0, 2],
  ['cont5', 2, 3],
  ['cont9_inf', 0, 4],
  ['cont10', 4, 3],
  ['cont6', 3, 5],
  ['bobina1', 5, 6],
  ['cont7', 3, 7],
  ['cont8', 7, 8],
  ['bobina2', 8, 9]
]


In [9]:
def ordenar_tipo_elemento(dados):
    #Com a lista ja ordenada, agora vou ordenar o elemento para colocalo na part do meu xml
    contagem = Counter([x[3] for x in dados])
    vistos = {}
    resultado = []

    for item in dados:
        valor = item[3]

        # conta quantas vezes já vi esse valor
        vistos[valor] = vistos.get(valor, 0) + 1

        resultado.append(item)

        # se for a ÚLTIMA ocorrência e tiver repetição
        if contagem[valor] > 1 and vistos[valor] == contagem[valor]:
            resultado.append(['card', contagem[valor]])

    return resultado

def difinir_uid_part(lista_ordenada_com_card,dados):
    '''
    O objetivo dessa funcao é adicionar uma lista com um ou dois elementos que serao os os UIds do nome do elemento e do tipo do elemento.
    Se for uma junção de cardinalidade, ele só recebera um elemento para ser colocado ao part   
    '''
    tamanho_lista =  len(lista_ordenada_com_card)
    tamanho_card = len(dados)
    uid_inicial = 21
    uid_part =  21 + tamanho_card
    for x in range(len(lista_ordenada_com_card)):
        if len(lista_ordenada_com_card)[x] == 2:
            lista_ordenada_com_card[x].append([])
            lista_ordenada_com_card[x][-1].append([uid_part])
            uid_part +=1  
        else:
            lista_ordenada_com_card[x].append([])
            lista_ordenada_com_card[x][-1].append([uid_inicial])
            lista_ordenada_com_card[x][-1].append([uid_part])
            uid_inicial +=1
            uid_part +=1
    uid_inicial = 21
    for x in range(len(dados)):
        dados[x].append([])
        dados[x][-1].append(uid_inicial)
        uid_inicial +=1

    return(lista_ordenada_com_card,dados,uid_part)



In [ ]:
lista_nomear = []
lista_de = []#de = definir elemento
dicionario_elementos = {"bobina":"coil","contato":"Contact"}
dicionario_bobinas = {"1":"coil","c":"coil","r":"RCoil","s":"SCoil"}
dicionario_contatos = {"1":"Contact","c":"contact"}
dicionario_elementos_completo = {"Contact":dicionario_contatos,"coil":dicionario_bobinas}

def nomear_elemento(uid,nome,tipo,subtipo='1',extra = 1):
    '''Cria o codigo xml para o nome do elemento'''
    access = ET.Element("Access", Scope ="GlobalVariable", UId = str(uid) )
    symbol = ET.SubElement(access, "Symbol")
    ET.SubElement(symbol, "Component", Name=str(nome))
    ET.indent(access, space="    ", level=0)
    xml_string = ET.tostring(access, encoding='unicode')
    print(xml_string)
    lista_nomear.append(xml_string)
    return('Resolvido')

def retornar_string_elemento(tipo,subtipo = '1'):
    string_elemento = dicionario_elementos_completo[dicionario_elementos[tipo]][subtipo]
    return(string_elemento)

def definir_elemento(uid,nome, tipo,subtipo = '1'):
    '''Cria o codigo xml para o tipo de elemento associado ao nome'''
    part = ET.Element("Part", Name = retornar_string_elemento(tipo,subtipo), UId = str(uid))
    xml_string_2 = ET.tostring(part, encoding='unicode')
    #lista definir
    lista_de.append(xml_string_2)
    print(xml_string_2)

def parte_esquerda(uid,numero_cardinalidade):
    '''
    Cria mais um #part referenciando ao objeto O de adição de elementos em um mesmo no
    Responsavel pela cardinalidade ao definir parts
    ATENCAO: PRECISO DEFINIR A  ENTRADA NUMERO_CARDINALIDADE QUE VEM DE UMA OUTRA LISTA QUE EU JA FIZ
    '''
    part = ET.Element("Part", Name="O", UId=str(uid)
    template_value = ET.SubElement(part, "TemplateValue", Name="Card", Type="Cardinality")
    template_value.text = str(numero_cardinalidade))
    ET.indent(part, space="    ", level=0)
    xml_string = ET.tostring(part, encoding='unicode')
    lista_de.append(xml_string)
    return(no_conexao)





In [8]:
x = [1,2,3]
len(x)

3

In [ ]:
#Bloco criação do elemento no texto
lista_uid = [] 
lista_dicionarios =[]
lista_nomear = []# lista com os nomes de cada elemento
lista_de = [] #lista com a declaracao em ingles de cada elemento
dicionario_elementos = {"bobina":"coil","contato":"Contact"}
dicionario_bobinas = {"1":"coil","c":"coil","r":"RCoil","s":"SCoil"}
dicionario_contatos = {"1":"Contact","c":"contact"}
dicionario_elementos_completo = {"Contact":dicionario_contatos,"coil":dicionario_bobinas}
dicionario_nomear_temporario ={}
dicionario_nomear={}
dicionario_nomear_invertido = {}
dicionario_casal_elementos ={}
lista_dicionarios_invertido = {}
dicionario_casal_invertido = {}
dicionario_nome_elemento = {}
lista_esquerda =[]
lista_uid.append(20)
def nomear_elemento(nome, tipo,subtipo="1",extra= 1 ):
    lista_uid.append(lista_uid[-1]+1)
    access = ET.Element("Access", Scope ="GlobalVariable", UId = str(lista_uid[-1]) )
    symbol = ET.SubElement(access, "Symbol")
    ET.SubElement(symbol, "Component", Name=str(nome))
    ET.indent(access, space="    ", level=0)
    xml_string = ET.tostring(access, encoding='unicode')
    dicionario_nomear_temporario ={str(lista_uid[-1]):str(nome)}
    dicionario_nomear.update(dicionario_nomear_temporario)#criando um dicionario com um nome e a UId do nome que eu criei
    dicionario_nomear_temporario = {valor: chave for chave, valor in dicionario_nomear_temporario.items()}
    dicionario_nomear_invertido.update(dicionario_nomear_temporario)
    print(xml_string)
    lista_nomear.append(xml_string)
    return(dicionario_nomear)

def retornar_string_elemento(tipo,subtipo = '1'):
    string_elemento = dicionario_elementos_completo[dicionario_elementos[tipo]][subtipo]
    return(string_elemento)

def definir_elemento(nome, tipo,subtipo = '1'):
    lista_uid.append(lista_uid[-1]+1)
    part = ET.Element("Part", Name = retornar_string_elemento(tipo,subtipo), UId = str(lista_uid[-1]))
    xml_string_2 = ET.tostring(part, encoding='unicode')
    #lista definir
    lista_de.append(xml_string_2)
    dicionario_casal_temporario = {str(lista_uid[-1]):str(lista_uid[-2])}
    dicionario_casal_elementos.update(dicionario_casal_temporario)
    dicionario_casal_temporario = {valor: chave for chave, valor in dicionario_casal_temporario.items()}
    dicionario_casal_invertido.update(dicionario_casal_temporario)
    tipo = tipo
    print(xml_string_2)

def elemento_completo(tipo,subtipo):
    tipo = tipo
    subtipo = subtipo
    ultima_chave_nome = next(reversed(dicionario_nomear))
    ultimo_valor_nome = dicionario_nomear[ultima_chave_nome]
    ultima_chave_elemento = next(reversed(dicionario_casal_elementos))
    ultimo_valor_elemento = dicionario_casal_elementos[ultima_chave_elemento]
    dicionario_nome_elemento.update({(ultimo_valor_nome,tipo,subtipo):[ultima_chave_nome,ultima_chave_elemento]})

def parte_esquerda(obj1):
    #Cria mais um #part referenciando ao objeto O de adição de elementos em um mesmo no
    lista_uid.append(lista_uid[-1]+1)
    no_conexao = lista_uid[-1]
    part = ET.Element("Part", Name="O", UId=str(lista_uid[-1]))
    template_value = ET.SubElement(part, "TemplateValue", Name="Card", Type="Cardinality")
    template_value.text = str(len(obj1))
    ET.indent(part, space="    ", level=0)
    xml_string = ET.tostring(part, encoding='unicode')
    lista_esquerda.append(xml_string)
    return(no_conexao)

def criar_elemento(nome,tipo,subtipo= "1",extra=1):
    nomear_elemento(nome,tipo,subtipo,1)
    definir_elemento(nome,tipo,subtipo)
    elemento_completo(tipo,subtipo)